<a href="https://colab.research.google.com/github/doa-2026/machine-learning/blob/main/project5%2B6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Data

In [47]:

path="/content/drive/MyDrive/sales_predictions_2023 (2).csv"
import pandas as pd
df=pd.read_csv(path)
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [48]:
df.duplicated().sum()


np.int64(0)

In [49]:
df.dtypes

,0
Item_Identifier,object
Item_Weight,float64
Item_Fat_Content,object
Item_Visibility,float64
Item_Type,object
Item_MRP,float64
Outlet_Identifier,object
Outlet_Establishment_Year,int64
Outlet_Size,object
Outlet_Location_Type,object


In [50]:
df["Item_Fat_Content"].value_counts()
df["Item_Fat_Content"]=df["Item_Fat_Content"].replace({"LF":"Low Fat","low fat":"Low Fat", "reg":"Regular"})
df["Item_Fat_Content"].value_counts()

,count
Item_Fat_Content,
Low Fat,5517
Regular,3006


In [51]:
o=df.select_dtypes("object").columns
for i in o:
  print(i)
  print(df[i].value_counts())
  print ("\n")



Item_Identifier
Item_Identifier
FDW13    10
FDG33    10
FDX31     9
FDT07     9
NCY18     9
         ..
FDO33     1
FDK57     1
FDT35     1
FDN52     1
FDE52     1
Name: count, Length: 1559, dtype: int64


Item_Fat_Content
Item_Fat_Content
Low Fat    5517
Regular    3006
Name: count, dtype: int64


Item_Type
Item_Type
Fruits and Vegetables    1232
Snack Foods              1200
Household                 910
Frozen Foods              856
Dairy                     682
Canned                    649
Baking Goods              648
Health and Hygiene        520
Soft Drinks               445
Meat                      425
Breads                    251
Hard Drinks               214
Others                    169
Starchy Foods             148
Breakfast                 110
Seafood                    64
Name: count, dtype: int64


Outlet_Identifier
Outlet_Identifier
OUT027    935
OUT013    932
OUT035    930
OUT049    930
OUT046    930
OUT045    929
OUT018    928
OUT017    926
OUT010    555
OUT019    

In [52]:
df.isna().sum()

,0
Item_Identifier,0
Item_Weight,1463
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,2410
Outlet_Location_Type,0


In [53]:
df["Outlet_Identifier"]=df["Outlet_Identifier"].str.replace("OUT0"," ")
df["Outlet_Identifier"]=df["Outlet_Identifier"].astype("int")
df["Outlet_Identifier"].value_counts()

,count
Outlet_Identifier,
27,935
13,932
35,930
49,930
46,930
45,929
18,928
17,926
10,555


Import packages

In [95]:
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor ,BaggingRegressor

Train - Test Split

In [55]:
X=df.drop(columns="Item_Outlet_Sales").drop(columns="Item_Identifier")
y=df["Item_Outlet_Sales"]
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42)
X_train.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
4776,16.350,Low Fat,0.029565,Household,256.4646,18,2009,Medium,Tier 3,Supermarket Type2
7510,15.250,Regular,0.000000,Snack Foods,179.7660,18,2009,Medium,Tier 3,Supermarket Type2
5828,12.350,Regular,0.158716,Meat,157.2946,49,1999,Medium,Tier 1,Supermarket Type1
5327,7.975,Low Fat,0.014628,Baking Goods,82.3250,35,2004,Small,Tier 2,Supermarket Type1
4810,19.350,Low Fat,0.016645,Frozen Foods,120.9098,45,2002,NaN,Tier 2,Supermarket Type1


In [56]:
X_train.dtypes

,0
Item_Weight,float64
Item_Fat_Content,object
Item_Visibility,float64
Item_Type,object
Item_MRP,float64
Outlet_Identifier,int64
Outlet_Establishment_Year,int64
Outlet_Size,object
Outlet_Location_Type,object
Outlet_Type,object


Create preprocessing Column Transformer

In [57]:
num_col=X_train.select_dtypes("number").columns
num_imp=SimpleImputer(strategy="mean")
scaler=StandardScaler()
num_pip=make_pipeline(num_imp,scaler)
tup_num=("number",num_pip,num_col)
tup_num
num_col

Index(['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year'],
      dtype='object')

In [58]:
cat_col=X_train.select_dtypes("object").columns.drop("Outlet_Size")
cat_imp=SimpleImputer(strategy="constant",fill_value="Missing")
cat_enc=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
cat_pip=make_pipeline(cat_imp,cat_enc)
cat_tup=("cat",cat_pip,cat_col)
cat_tup
cat_col

Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Location_Type', 'Outlet_Type'], dtype='object')

In [66]:
ord=["Outlet_Size"]
order=["Small","Medium","High"]
size_imp=SimpleImputer(strategy="most_frequent")
size_enc=OrdinalEncoder(categories=[order])
scale=StandardScaler()
size_pip=make_pipeline(size_imp,size_enc,scale)
ord_tup=("ordinal",size_pip,ord)
ord_tup
size_pip

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('ordinalencoder',
                 OrdinalEncoder(categories=[['Small', 'Medium', 'High']])),
                ('standardscaler', StandardScaler())])

In [67]:
col_transformer=ColumnTransformer([tup_num,cat_tup,ord_tup],verbose_feature_names_out=False)
col_transformer.fit(X_train)

ColumnTransformer(transformers=[('number',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('oneho...
                                                                sparse_output=False))]),
                                 Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Location_Type', 'Outlet_Type'], dtype='object')),
                                ('ordinal',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(categories=[['Small',
                                                                              'Medium',
                                                                              'High']])),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 ['Outlet_Size'])],
                  verbose_feature_names_out=False)

In [85]:
col_transformer.fit(X_train)
X_train_processing=col_transformer.transform(X_train)
X_test_processing=col_transformer.transform(X_test)
featuer_new=col_transformer.get_feature_names_out()
X_train_processing.head()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,...,Item_Type_Soft Drinks,Item_Type_Starchy Foods,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Outlet_Size
4776,0.817249,-0.712775,1.828109,-0.798503,1.327849,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.287374
7510,0.556340,-1.291052,0.603369,-0.798503,1.327849,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.287374
5828,-0.131512,1.813319,0.244541,1.437260,0.136187,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.287374
5327,-1.169219,-1.004931,-0.952591,0.427561,0.732018,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-1.384048
4810,1.528819,-0.965484,-0.336460,1.148775,0.493686,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.287374


Import and Instantiate the Model (linear Regestion)

In [79]:
lin_reg =LinearRegression()
lin_reg.fit(X_train_processing,y_train)


LinearRegression()

In [86]:
lin_reg.intercept_
lin_reg.coef_
coef=pd.Series(lin_reg.coef_,index=featuer_new)
coef

,0
Item_Weight,-7.595954
Item_Visibility,-21.621302
Item_MRP,984.524728
Outlet_Identifier,-52.670877
Outlet_Establishment_Year,18.793406
Item_Fat_Content_Low Fat,-31.088644
Item_Fat_Content_Regular,31.088644
Item_Type_Baking Goods,-19.586888
Item_Type_Breads,-44.011332
Item_Type_Breakfast,28.257926


Evalute the Model

In [89]:
y_pre_test=lin_reg.predict(X_test_processing)
y_pre_train=lin_reg.predict(X_train_processing)

In [92]:
 #Add custom function

def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

In [94]:
evaluate_regression(lin_reg, X_train_processing, y_train, X_test_processing, y_test,output_frame=True)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 847.267
- MSE = 1,299,657.005
- R^2 = 0.561

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 804.491
- MSE = 1,195,727.326
- R^2 = 0.567


,MAE,MSE,R^2
Training Data,847.267,1299657.005,0.561
Test Data,804.491,1195727.326,0.567


tis value of R squared isn't higher  for traning , testing  ....

it's seem like underfitting

define Random Forest

In [ ]:
rf=RandomForestRegressor(random_state=42)
rf_pip=make_pipeline(col_transformer,rf)
rf_pip.fit(X_train,y_train)

In [97]:
evaluate_regression(rf_pip, X_train, y_train, X_test, y_test,output_frame=True)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 297.835
- MSE = 183,963.304
- R^2 = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 768.944
- MSE = 1,229,355.737
- R^2 = 0.554


,MAE,MSE,R^2
Training Data,297.835,183963.304,0.938
Test Data,768.944,1229355.737,0.554



the R squred in Training data is better than Testing data
its overfitting

the Random Forest is overfitting it need to make hyperparameter tunning to improve the model

the liner Regerstion model is underfitting which is not good in training and testing data

In [104]:
rf_pip.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('number',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer()),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Identifier',
          'Outlet_Establishment_Year'],
         dtype='object')),
                                   ('cat',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(fill_value='Missing',
                                                                   strategy='constant')),
                                                    ('oneho...
                                                                   sparse_output=False

In [118]:
from sklearn.model_selection import GridSearchCV
param_grid= {"randomforestregressor__max_depth":[2,4,6,9], "randomforestregressor__min_samples_leaf":[2,5,7,9]}
Grid_search=GridSearchCV(rf_pip,param_grid,n_jobs=-1,verbose=1)

In [119]:
Grid_search.fit(X_train,y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(estimator=Pipeline(steps=[('columntransformer',
                                        ColumnTransformer(transformers=[('number',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer()),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Item_Weight', 'Item_Visibility', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          Simp...
                                                                                         ('ordinalencoder',
                                                                                          OrdinalEncoder(categories=[['Small',
                                                                                                                      'Medium',
                                                                                                                      'High']])),
                                                                                         ('standardscaler',
                                                                                          StandardScaler())]),
                                                                         ['Outlet_Size'])],
                                                          verbose_feature_names_out=False)),
                                       ('randomforestregressor',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'randomforestregressor__max_depth': [2, 4, 6, 9],
                         'randomforestregressor__min_samples_leaf': [2, 5, 7,
                                                                     9]},
             verbose=1)

In [120]:
Grid_search.best_params_

{'randomforestregressor__max_depth': 6,
 'randomforestregressor__min_samples_leaf': 2}

In [121]:
best_model=Grid_search.best_estimator_
evaluate_regression(best_model, X_train, y_train, X_test, y_test,output_frame=True)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 742.313
- MSE = 1,112,526.580
- R^2 = 0.624

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 727.599
- MSE = 1,101,334.554
- R^2 = 0.601


,MAE,MSE,R^2
Training Data,742.313,1112526.580,0.624
Test Data,727.599,1101334.554,0.601


The tunned model is much better , the r squred become .6  in  both training and testing data , the performance improve

I recommended to used Tunning Random forest  ... this model is not underfit neither  overfit


this model can account of about  60% of the variance in y-test using x-test


 MAe (the averge difference between the acutal item sales and preducted value  ) is 727  its acceptable  .. it easy to understand
